<a href="https://colab.research.google.com/github/Juan-Medinaa/Inferencia/blob/main/BayesianModelFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo Jerárquico Bayesiano de Poisson para la Predicción de Goles y Evaluación de Valor Esperado (Bundesliga)

Este proyecto desarrolla e implementa un **Modelo Jerárquico Bayesiano basado en la distribución de Poisson** para pronosticar los resultados de fútbol de la Bundesliga.

El objetivo financiero del modelo es contrastar sus probabilidades calculadas frente a las cuotas del mercado (*Bet365 / Pinnacle*) para identificar **Valor Esperado Positivo ($EV+$)**, aplicando simulaciones de Montecarlo y criterios de gestión de capital (*Flat Stake*) para evaluar la viabilidad y rentabilidad del modelo en el mercado de **Goles (Over/Under 2.5)**.

## 1. Configuración del Entorno e Importación de Librerías



In [ ]:
import sys
!{sys.executable} -m pip install understatapi tqdm matplotlib scikit-learn pandas numpy

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="PyTensor could not link to a BLAS")

from understatapi import UnderstatClient
import pandas as pd
import numpy as np
from time import sleep
from tqdm.auto import tqdm
from pprint import pprint

import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from scipy.stats import poisson

import ipywidgets as widgets
from IPython.display import display

## 2. Extracción de Datos Históricos de Rendimiento Deportivo

En esta sección se define el alcance temporal del proyecto y se extrae secuencialmente el histórico de partidos de la Bundesliga a través de la API, consolidando una base de datos multi-temporada que servirá como el conjunto de entrenamiento para estimar los parámetros latentes del modelo bayesiano.

In [ ]:
# Configuración
LEAGUE = "Bundesliga"    # "EPL", "La_Liga", "Bundesliga", "Serie_A", "Ligue_1", "RFPL"
SEASON = ["2021","2022", "2023", "2024", "2025"]

In [ ]:
def get_league_matches(league: str, seasons: list[str]):
    all_matches = []
    with UnderstatClient() as us:
        for season in seasons:
            matches = us.league(league=league).get_match_data(season=season)
            all_matches.extend(matches)
    return all_matches

raw_matches = get_league_matches(LEAGUE, SEASON)
print(f"Total de partidos: {len(raw_matches)}")
pprint(raw_matches[0])

## 3. Estructuración, Etiquetado y Codificación de Variables del Dataset

En esta sección se limpian los datos en bruto filtrando únicamente los partidos concluidos, se clasifican los resultados observados ($H$: Local, $A$: Visitante, $D$: Empate) y se indexan algebraicamente los equipos mediante valores numéricos enteros ($0$ a $N-1$).

In [ ]:
def format_fixtures(matches: list[dict]) -> pd.DataFrame:
    rows = []
    for m in matches:
        if not m.get("isResult"):
            continue

        h_goals = int(m["goals"]["h"])
        a_goals = int(m["goals"]["a"])

        if h_goals > a_goals:
            result = "H"
        elif h_goals < a_goals:
            result = "A"
        else:
            result = "D"

        rows.append({
            "date": m["datetime"],
            "home_team": m["h"]["title"],
            "away_team": m["a"]["title"],
            "yg1": h_goals,
            "yg2": a_goals,
            "result": result
        })

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)

fixtures = format_fixtures(raw_matches)
print(f"Partidos jugados: {len(fixtures)}")
fixtures.head()

In [ ]:
n_teams = len(fixtures["home_team"].unique())

teams = (
    fixtures[["home_team"]]
    .drop_duplicates()
    .sort_values("home_team")
    .reset_index(drop=True)
    .assign(team_index=np.arange(n_teams))
    .rename(columns={"home_team": "team"})
)

df = (
    fixtures
    .merge(teams, left_on="home_team", right_on="team")
    .rename(columns={"team_index": "hg"})
    .drop(["team"], axis=1)
    .merge(teams, left_on="away_team", right_on="team")
    .rename(columns={"team_index": "ag"})
    .drop(["team"], axis=1)
    .sort_values("date")
)

print(f"Equipos: {n_teams}")
print(teams.to_string())
print()
df.head()

## 4. Particionado de Datos y Especificación del Modelo Jerárquico Bayesiano

En esta sección se dividen los datos en conjuntos de entrenamiento y prueba (reservando los últimos 306 partidos correspondientes a la temporada de validación 2025/2026) y se define la arquitectura matemática del modelo jerárquico empleando las siguientes ecuaciones log-lineales para los parámetros de intensidad $\theta$:

$$\log(\theta_{home}) = \text{home} + \text{atts}_{home\_team} + \text{defs}_{away\_team}$$
$$\log(\theta_{away}) = \text{atts}_{away\_team} + \text{defs}_{home\_team}$$

### Lógica Conceptual del Modelo:
* **Estructura Jerárquica:** Las habilidades de ataque ($\text{atts_star}$) y defensa ($\text{def_star}$) se modelan con distribuciones normales condicionadas por hiperparámetros de precisión $\tau$ (distribuciones Gamma), permitiendo que los equipos compartan información estadística mutua (*shrinkage*).
* **Restricción de Identificabilidad:** Se aplica una transformación determinista restando la media ($\text{atts_star} - \mu_{atts_star}$) para centrar los parámetros en cero. Esto evita la redundancia algebraica y asegura que las habilidades individuales se interpreten rigurosamente respecto al promedio de la liga.

* **Función de Verosimilitud:** Los goles observados se asumen bajo una distribución de Poisson condicionalmente independiente basada en los valores exponenciados de $\theta_{home}$ y $\theta_{away}$.

In [ ]:
TEST_SIZE = 306

train = df.iloc[:-TEST_SIZE]
test = df.iloc[-TEST_SIZE:]

goals_home_obs = train["yg1"].values
goals_away_obs = train["yg2"].values
home_team = train["hg"].values
away_team = train["ag"].values

print(f"Train: {len(train)} partidos")
print(f"Test: {len(test)} partidos")

In [ ]:
import pytensor.tensor as pt

with pm.Model() as model:
    home = pm.Flat("home")  # Prior No Informativo de la ventaja de local

    tau_att = pm.Gamma("tau_att", 0.1, 0.1) # Hiperprior Gamma para precisión de ataque
    atts_star = pm.Normal("atts_star", mu=0, tau=tau_att, shape=n_teams) # Prior condicional Normal

    tau_def = pm.Gamma("tau_def", 0.1, 0.1)  # Hiperprior Gamma para precisión de defensa
    def_star = pm.Normal("def_star", mu=0, tau=tau_def, shape=n_teams) # Prior condicional Normal

    atts = pm.Deterministic("atts", atts_star - pt.mean(atts_star))
    defs = pm.Deterministic("defs", def_star - pt.mean(def_star))

    home_theta = pt.exp(home + atts[home_team] + defs[away_team])
    away_theta = pt.exp(atts[away_team] + defs[home_team])

    home_goals = pm.Poisson("home_goals", mu=home_theta, observed=goals_home_obs)
    away_goals = pm.Poisson("away_goals", mu=away_theta, observed=goals_away_obs)

## 5. Muestreo de la Distribución Posterior mediante MCMC

En esta sección se ejecuta el proceso de inferencia bayesiana utilizando algoritmos de Montecarlo por Cadenas de Markov (MCMC). El sampler genera 2,000 muestras de la distribución posterior tras una fase previa de 1,000 pasos de ajuste (*tuning*) por cada una de las 4 cadenas en paralelo, guardando los resultados en un objeto de inferencia estructurado para evaluar la convergencia y la incertidumbre de los parámetros de ataque y defensa.

In [ ]:
with model:
    trace = pm.sample(2000, tune=1000, cores=4, return_inferencedata=True)

## 6. Diagnóstico Visual de la Convergencia de Parámetros

En esta sección se generan los gráficos de traza para evaluar visualmente la convergencia de las cadenas MCMC, tomando como referencia el parámetro del efecto de localía (`home`). El gráfico permite verificar la correcta superposición de las densidades estimadas (izquierda) y la ausencia de tendencias o patrones anómalos en el recorrido de los muestreadores a lo largo de las iteraciones (derecha).

In [ ]:
az.plot_trace(trace, var_names=["Home"])
plt.tight_layout()

## 7. Extracción de Métricas Posteriores y Visualización de Intervalos de Credibilidad (HDI)

En esta sección se calculan las medianas de las habilidades marginales y sus respectivos Intervalos de Densidad Más Alta al 94% ($HDI$) para el ataque y la defensa de cada equipo. La visualización mediante gráficos de barras de error permite contrastar la magnitud de los parámetros latentes ordenados jerárquicamente, haciendo evidente la incertidumbre epistémica (amplitud de las barras) asociada al rendimiento verdadero de cada club de la Bundesliga.

In [ ]:
atts_samples = trace.posterior["atts"].values.reshape(-1, n_teams)

atts_df = (
    pd.DataFrame(az.hdi(trace, var_names=["atts"])["atts"].values, columns=["lower_hdi", "upper_hdi"])
    .assign(median=np.median(atts_samples, axis=0))
    .merge(teams, left_index=True, right_on="team_index")
    .drop(["team_index"], axis=1)
    .rename(columns={"team": "Team"})
    .assign(lower=lambda x: x["median"] - x["lower_hdi"])
    .assign(upper=lambda x: x["upper_hdi"] - x["median"])
    .sort_values("median", ascending=True)
)

plt.figure(figsize=(8, 10))
plt.errorbar(atts_df["median"], atts_df["Team"], xerr=(atts_df[["lower", "upper"]].values).T, fmt="o")
plt.xlabel("Clasificación de Ataque")
plt.title("Calificacion de Ataque por Equipo")
plt.tight_layout()

In [ ]:
defs_samples = trace.posterior["defs"].values.reshape(-1, n_teams)

defs_df = (
    pd.DataFrame(az.hdi(trace, var_names=["defs"])["defs"].values, columns=["lower_hdi", "upper_hdi"])
    .assign(median=np.median(defs_samples, axis=0))
    .merge(teams, left_index=True, right_on="team_index")
    .drop(["team_index"], axis=1)
    .rename(columns={"team": "Team"})
    .assign(lower=lambda x: x["median"] - x["lower_hdi"])
    .assign(upper=lambda x: x["upper_hdi"] - x["median"])
    .sort_values("median", ascending=False)
)

plt.figure(figsize=(8, 10))
plt.errorbar(defs_df["median"], defs_df["Team"], xerr=(defs_df[["lower", "upper"]].values).T, fmt="o")
plt.xlabel("Calificación de Defensa")
plt.title("Calificacion Defensivas por Equipo")
plt.tight_layout()

## 8. Funciones de Predicción y Construcción de la Matriz de Probabilidades Binomial-Poisson (1X2)

En esta sección se definen las funciones core para proyectar partidos: la primera colapsa las distribuciones posteriores mediante su promedio para obtener los parámetros de intensidad esperados ($\theta_{home}, \theta_{away}$), y la segunda aplica el producto externo de sus Funciones de Masa de Probabilidad ($PMF$) para construir una matriz estocástica bidimensional, resolviendo las probabilidades del mercado 1X2 mediante operaciones algebraicas sobre sus sectores corporales (triángulo inferior para victoria local, diagonal para empate y triángulo superior para victoria visitante).

> **Nota Crítica de Diseño:** El uso de `.mean()` en esta etapa destruye la incertidumbre útil de la distribución posterior bayesiana antes de calcular las probabilidades, lo que introduce un sesgo estructural que reduce la precisión del modelo frente a la eficiencia del mercado.

In [ ]:
def goal_expectation(trace, home_team_id, away_team_id):
    home = trace.posterior["home"].values.mean()
    atts = trace.posterior["atts"].values.reshape(-1, n_teams)
    defs = trace.posterior["defs"].values.reshape(-1, n_teams)

    atts_home = atts[:, home_team_id].mean()
    atts_away = atts[:, away_team_id].mean()
    defs_home = defs[:, home_team_id].mean()
    defs_away = defs[:, away_team_id].mean()

    home_theta = np.exp(home + atts_home + defs_away)
    away_theta = np.exp(atts_away + defs_home)

    return home_theta, away_theta

In [ ]:
def win_draw_loss(home_expectation, away_expectation, max_goals=10):
    h = poisson.pmf(range(max_goals + 1), home_expectation)
    a = poisson.pmf(range(max_goals + 1), away_expectation)
    m = np.outer(h, a)

    home_win = np.sum(np.tril(m, -1))
    away_win = np.sum(np.triu(m, 1))
    draw = np.sum(np.diag(m))

    return home_win, draw, away_win

## 9. Modelado Multi-Mercado Basado en la Matriz de Probabilidad Conjunta de Poisson

En esta sección se calcula la matriz estocástica de marcadores exactos mediante el producto externo de las probabilidades de Poisson individuales, permitiendo derivar analíticamente las cuotas justas para mercados complejos y secundarios como Over/Under (Totales y por Equipo), Ambos Marcan (BTTS), Doble Oportunidad y Hándicaps Asiáticos (incluyendo líneas fraccionadas de cuartos y mitades).

In [ ]:
def get_score_matrix(home_exp, away_exp, max_goals=10):
    """Genera la matriz de probabilidades de resultados exactos."""
    h = poisson.pmf(range(max_goals + 1), home_exp)
    a = poisson.pmf(range(max_goals + 1), away_exp)
    return np.outer(h, a)

def over_under_goals(home_exp, away_exp, line=2.5, max_goals=10):
    """
    Calcula probabilidades Over/Under para total de goles.
    line: 0.5, 1.5, 2.5, 3.5, etc.
    """
    m = get_score_matrix(home_exp, away_exp, max_goals)

    over_prob = 0
    under_prob = 0

    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            total = i + j
            if total > line:
                over_prob += m[i, j]
            else:
                under_prob += m[i, j]

    return {"over": over_prob, "under": under_prob}

def btts(home_exp, away_exp, max_goals=10):
    """
    Both Teams To Score (Ambos equipos marcan).
    """
    m = get_score_matrix(home_exp, away_exp, max_goals)

    # BTTS Yes: ambos marcan al menos 1 (excluir fila 0 y columna 0)
    btts_yes = np.sum(m[1:, 1:])
    btts_no = 1 - btts_yes

    return {"yes": btts_yes, "no": btts_no}

def over_under_team(team_exp, line=0.5, max_goals=10):
    """
    Over/Under goles para un equipo específico.
    """
    probs = poisson.pmf(range(max_goals + 1), team_exp)

    over_prob = sum(probs[int(line) + 1:])
    under_prob = sum(probs[:int(line) + 1])

    return {"over": over_prob, "under": under_prob}

def double_chance(home_exp, away_exp, max_goals=10):
    """
    Doble oportunidad: 1X, X2, 12
    """
    h_win, draw, a_win = win_draw_loss(home_exp, away_exp, max_goals)

    return {
        "1X": h_win + draw,      # Local gana o empata
        "X2": draw + a_win,      # Empata o visitante gana
        "12": h_win + a_win      # Local o visitante gana (no empate)
    }

def asian_handicap_line(home_exp, away_exp, line, max_goals=10):
    """
    Calcula handicap asiático para una línea específica.
    line: handicap del local (negativo = desventaja, positivo = ventaja)

    Líneas enteras (.0): win/push/lose
    Líneas medias (.5): win/lose (sin push)
    Líneas cuarto (.25, .75): split bet entre dos líneas adyacentes

    Ejemplos:
    - line = -0.25: split entre 0 y -0.5
    - line = +0.25: split entre 0 y +0.5
    - line = -0.75: split entre -0.5 y -1
    - line = +0.75: split entre +0.5 y +1
    """
    m = get_score_matrix(home_exp, away_exp, max_goals)

    # Detectar tipo de línea
    decimal_part = round(abs(line) % 1, 2)

    if decimal_part in [0.25, 0.75]:
        # Split bet: promedio de dos líneas adyacentes
        if decimal_part == 0.25:
            # .25 está entre .0 y .5
            if line >= 0:
                line1 = np.floor(line)      # ej: +0.25 -> 0
                line2 = line1 + 0.5         # ej: +0.25 -> +0.5
            else:
                line1 = np.ceil(line)       # ej: -0.25 -> 0
                line2 = line1 - 0.5         # ej: -0.25 -> -0.5
        else:  # 0.75
            # .75 está entre .5 y siguiente entero
            if line >= 0:
                line1 = np.floor(line) + 0.5  # ej: +0.75 -> +0.5
                line2 = line1 + 0.5           # ej: +0.75 -> +1
            else:
                line1 = np.ceil(line) - 0.5   # ej: -0.75 -> -0.5
                line2 = line1 - 0.5           # ej: -0.75 -> -1

        result1 = asian_handicap_line(home_exp, away_exp, line1, max_goals)
        result2 = asian_handicap_line(home_exp, away_exp, line2, max_goals)

        # Promedio ponderado (50% cada línea)
        home_prob = (result1["home"] + result2["home"]) / 2
        away_prob = (result1["away"] + result2["away"]) / 2

        return {"home": home_prob, "away": away_prob}

    else:
        # Línea entera o media
        home_win = 0
        away_win = 0
        push = 0

        for i in range(max_goals + 1):
            for j in range(max_goals + 1):
                adjusted_diff = (i + line) - j

                if adjusted_diff > 0:
                    home_win += m[i, j]
                elif adjusted_diff < 0:
                    away_win += m[i, j]
                else:
                    push += m[i, j]

        # Para líneas .5, no hay push
        if decimal_part == 0.5:
            return {"home": home_win, "away": away_win}
        else:
            # Línea entera: push devuelve apuesta
            # Calcular probabilidades excluyendo push (draw no bet style)
            total = home_win + away_win
            if total > 0:
                return {"home": home_win / total, "away": away_win / total, "push": push}
            else:
                return {"home": 0.5, "away": 0.5, "push": push}

def asian_handicap_table(home_exp, away_exp, max_goals=10):
    """
    Genera tabla completa de handicap asiático.
    Retorna dict con todas las líneas desde -2.5 hasta +2.5
    """
    lines = [-2.5, -2.25, -2, -1.75, -1.5, -1.25, -1, -0.75, -0.5, -0.25,
             0, 0.25, 0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5]

    results = {}
    for line in lines:
        ah = asian_handicap_line(home_exp, away_exp, line, max_goals)
        results[line] = {
            "home_prob": ah["home"],
            "away_prob": ah["away"],
            "home_odds": 1 / ah["home"] if ah["home"] > 0.001 else 999,
            "away_odds": 1 / ah["away"] if ah["away"] > 0.001 else 999
        }

    return results


In [ ]:
def predict_match(trace, home_team_name, away_team_name):
    """Genera predicción completa para un partido con todos los mercados."""
    home_id = teams[teams["team"] == home_team_name]["team_index"].values[0]
    away_id = teams[teams["team"] == away_team_name]["team_index"].values[0]

    h_exp, a_exp = goal_expectation(trace, home_id, away_id)

    # 1X2
    h_win, draw, a_win = win_draw_loss(h_exp, a_exp)

    # Over/Under totales
    ou_05 = over_under_goals(h_exp, a_exp, 0.5)
    ou_15 = over_under_goals(h_exp, a_exp, 1.5)
    ou_25 = over_under_goals(h_exp, a_exp, 2.5)
    ou_35 = over_under_goals(h_exp, a_exp, 3.5)

    # BTTS
    btts_prob = btts(h_exp, a_exp)

    # Over/Under por equipo
    home_ou_05 = over_under_team(h_exp, 0.5)
    home_ou_15 = over_under_team(h_exp, 1.5)
    away_ou_05 = over_under_team(a_exp, 0.5)
    away_ou_15 = over_under_team(a_exp, 1.5)

    # Doble oportunidad
    dc = double_chance(h_exp, a_exp)

    # Handicap Asiático
    ah_table = asian_handicap_table(h_exp, a_exp)

    # Imprimir resultados
    print(f"{'='*60}")
    print(f"{home_team_name} vs {away_team_name}")
    print(f"Goles esperados: {h_exp:.2f} - {a_exp:.2f}")
    print(f"{'='*60}")

    print(f"\n--- 1X2 ---")
    print(f"{'Resultado':<12} {'Prob':>8} {'Cuota':>8}")
    print(f"{'Local':<12} {h_win:>7.1%} {1/h_win:>8.2f}")
    print(f"{'Empate':<12} {draw:>7.1%} {1/draw:>8.2f}")
    print(f"{'Visitante':<12} {a_win:>7.1%} {1/a_win:>8.2f}")

    print(f"\n--- Over/Under (Total) ---")
    print(f"{'Linea':<12} {'Over':>8} {'Under':>8} {'O.Cuota':>8} {'U.Cuota':>8}")
    print(f"{'0.5':<12} {ou_05['over']:>7.1%} {ou_05['under']:>7.1%} {1/ou_05['over']:>8.2f} {1/ou_05['under']:>8.2f}")
    print(f"{'1.5':<12} {ou_15['over']:>7.1%} {ou_15['under']:>7.1%} {1/ou_15['over']:>8.2f} {1/ou_15['under']:>8.2f}")
    print(f"{'2.5':<12} {ou_25['over']:>7.1%} {ou_25['under']:>7.1%} {1/ou_25['over']:>8.2f} {1/ou_25['under']:>8.2f}")
    print(f"{'3.5':<12} {ou_35['over']:>7.1%} {ou_35['under']:>7.1%} {1/ou_35['over']:>8.2f} {1/ou_35['under']:>8.2f}")

    print(f"\n--- BTTS (Ambos Marcan) ---")
    print(f"{'Si':<12} {btts_prob['yes']:>7.1%} {1/btts_prob['yes']:>8.2f}")
    print(f"{'No':<12} {btts_prob['no']:>7.1%} {1/btts_prob['no']:>8.2f}")

    print(f"\n--- Over/Under por Equipo ---")
    print(f"{home_team_name}:")
    print(f"  {'O 0.5':<10} {home_ou_05['over']:>7.1%} {1/home_ou_05['over']:>8.2f}")
    print(f"  {'U 0.5':<10} {home_ou_05['under']:>7.1%} {1/home_ou_05['under']:>8.2f}")
    print(f"  {'O 1.5':<10} {home_ou_15['over']:>7.1%} {1/home_ou_15['over']:>8.2f}")
    print(f"  {'U 1.5':<10} {home_ou_15['under']:>7.1%} {1/home_ou_15['under']:>8.2f}")
    print(f"{away_team_name}:")
    print(f"  {'O 0.5':<10} {away_ou_05['over']:>7.1%} {1/away_ou_05['over']:>8.2f}")
    print(f"  {'U 0.5':<10} {away_ou_05['under']:>7.1%} {1/away_ou_05['under']:>8.2f}")
    print(f"  {'O 1.5':<10} {away_ou_15['over']:>7.1%} {1/away_ou_15['over']:>8.2f}")
    print(f"  {'U 1.5':<10} {away_ou_15['under']:>7.1%} {1/away_ou_15['under']:>8.2f}")

    print(f"\n--- Doble Oportunidad ---")
    print(f"{'1X (Local/Empate)':<20} {dc['1X']:>7.1%} {1/dc['1X']:>8.2f}")
    print(f"{'X2 (Empate/Visit.)':<20} {dc['X2']:>7.1%} {1/dc['X2']:>8.2f}")
    print(f"{'12 (Sin empate)':<20} {dc['12']:>7.1%} {1/dc['12']:>8.2f}")

    print(f"\n--- Handicap Asiático ---")
    print(f"{'Linea':>8} {'Home':>12} {'Away':>12}")
    print(f"{'-'*34}")
    for line in sorted(ah_table.keys()):
        data = ah_table[line]
        line_str = f"{line:+.2f}" if line != 0 else "0"
        print(f"{line_str:>8} {data['home_odds']:>12.3f} {data['away_odds']:>12.3f}")

    return {
        "home_exp": h_exp,
        "away_exp": a_exp,
        "1x2": {"home": h_win, "draw": draw, "away": a_win},
        "ou_25": ou_25,
        "btts": btts_prob,
        "double_chance": dc,
        "asian_handicap": ah_table
    }

# Ejemplo de predicción
result = predict_match(trace, "Bayern Munich", "Borussia Dortmund")

## 10. Evaluación de Calidad Probabilística mediante el Ranked Probability Score (RPS)

En esta sección se implementa la métrica $RPS$ para evaluar la precisión y calibración de las predicciones categóricas ordinales ($1, X, 2$) del modelo, promediando las desviaciones cuadráticas acumuladas frente al resultado real en el conjunto de prueba mediante la ecuación:

$$RPS = \frac{1}{M - 1} \sum_{m=1}^{M-1} \left( \sum_{i=1}^{m} p_i - \sum_{i=1}^{m} o_i \right)^2$$

### Lógica Conceptual:
* **Penalización Ordinal:** A diferencia de la entropía cruzada (*Log Loss*), el $RPS$ castiga con mayor severidad a los pronósticos que se alejan de la realidad según el orden jerárquico del fútbol; por ejemplo, si el partido termina en victoria local ($H$), predecir un empate ($D$) tiene menor penalización que predecir un triunfo visitante ($A$).
* **Calidad de Calibración:** Un $RPS$ promedio más bajo indica que las distribuciones acumuladas de probabilidad del modelo reflejan con mayor fidelidad la frecuencia relativa de los resultados observados en la Bundesliga.

In [ ]:
def rps(predictions, outcome):
    cumulative_pred = np.cumsum(predictions)
    cumulative_actual = np.zeros(3)
    cumulative_actual[outcome:] = 1
    return np.sum((cumulative_pred - cumulative_actual) ** 2) / 2

def calculate_rps(trace, df):
    rps_list = []
    for idx, row in df.iterrows():
        if row["result"] == "H":
            outcome = 0
        elif row["result"] == "D":
            outcome = 1
        else:
            outcome = 2

        h, a = goal_expectation(trace, row["hg"], row["ag"])
        predictions = win_draw_loss(h, a)
        rps_list.append(rps(predictions, outcome))

    return np.mean(rps_list)

rps_score = calculate_rps(trace, test)
print(f"RPS promedio en test: {rps_score:.4f}")

## 11. Carga de Datos de Mercado y Segmentación de Cuotas Comerciales (Bet365 / Pinnacle)

En esta sección se adquiere el dataset externo con el histórico y cuotas de la Bundesliga 2025/2026, aislando las variables de información principal (*Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR*) y filtrando exclusivamente las cuotas del mercado 1X2 de las casas de apuestas más eficientes y representativas (*Bet365* y *Pinnacle* de cierre). Esto estructura la base de datos de validación comercial y omite mercados alternativos iniciales, preparando el entorno para realizar el *backtesting* comparativo contra las probabilidades del modelo bayesiano.

In [ ]:
# Construye la URL raw para el archivo D1.csv en GitHub
url = 'https://raw.githubusercontent.com/Juan-Medinaa/Inferencia/main/D1%20(1).csv'

# Carga el archivo CSV en un DataFrame de pandas
df_bundesliga = pd.read_csv(url)

# Muestra las primeras 5 filas del DataFrame
display(df_bundesliga.head())

In [ ]:
# Columnas de información principal que siempre se muestran
core_info_columns = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']

# Prefijos de casas de apuestas y sus nombres completos
bookmakers_names = {
    'B365': 'Bet365', 'BW': 'Bet&Win', 'WH': 'William Hill', 'VC': 'Victor Chandler',
    'GB': 'Gamebookers', 'BS': 'Blue Square', 'SJ': 'Stan James', 'LB': 'Ladbrokes',
    'SB': 'Sportingbet', 'PP': 'Paddy Power',
    'PS': 'Pinnacle (1X2 Cierre)', # Específico para las cuotas de cierre 1X2
    'P': 'Pinnacle (O/U & AH)', # Para Más/Menos y Hándicap Asiático
    'Max': 'Máximo', 'Avg': 'Promedio'
}

# Inicializar un diccionario para las opciones de mercado, comenzando con estadísticas generales
market_options = {
    'Estadísticas del Partido': ['HS', 'AS', 'HST', 'AST', 'HC', 'AC']
}

all_columns = df_bundesliga.columns

# Definir las casas de apuestas específicas a incluir
filtered_bookmaker_prefixes = ['B365', 'PS', 'P']

# Identificar cuotas 1X2 y agregarlas a market_options
one_x_two_suffixes = ['H', 'D', 'A']
for prefix in sorted(filtered_bookmaker_prefixes):
    cols_found = [f"{prefix}{suffix}" for suffix in one_x_two_suffixes if f"{prefix}{suffix}" in all_columns]
    if len(cols_found) == 3: # Asegurarse de que existan todas las columnas H, D, A para la casa de apuestas
        bookie_name = bookmakers_names.get(prefix, prefix)
        market_options[f'1X2 - {bookie_name}'] = cols_found

# Identificar cuotas de Más/Menos 2.5 y agregarlas a market_options
ou_suffixes = ['>2.5', '<2.5']
for prefix in sorted(filtered_bookmaker_prefixes):
    cols_found = [f"{prefix}{suffix}" for suffix in ou_suffixes if f"{prefix}{suffix}" in all_columns]
    if len(cols_found) == 2: # Asegurarse de que existan ambas columnas >2.5 y <2.5
        bookie_name = bookmakers_names.get(prefix, prefix)
        market_options[f'Goles Over/Under 2.5 - {bookie_name}'] = cols_found

# Identificar cuotas de Hándicap Asiático y agregarlas a market_options
ah_suffixes = ['AHH', 'AHA']
ah_common_col = 'AHCh' if 'AHCh' in all_columns else None

for prefix in sorted(filtered_bookmaker_prefixes):
    cols_found = [f"{prefix}{suffix}" for suffix in ah_suffixes if f"{prefix}{suffix}" in all_columns]
    if len(cols_found) == 2: # Asegurarse de que existan ambas columnas AHH y AHA
        bookie_name = bookmakers_names.get(prefix, prefix)
        if ah_common_col:
            market_options[f'Asian Handicap - {bookie_name}'] = [ah_common_col] + cols_found
        else:
            market_options[f'Asian Handicap - {bookie_name}'] = cols_found

def mostrar_mercado(mercado):
    # Siempre comenzar con las columnas de información principal
    columnas = list(core_info_columns)

    # Anexar columnas para la opción de mercado seleccionada
    if mercado in market_options:
        columnas.extend(market_options[mercado])

    print(f" Mostrando: {mercado}")
    display(df_bundesliga[columnas].head(10))

# Crear el menú desplegable con todas las opciones de mercado disponibles
selector = widgets.Dropdown(
    options=sorted(list(market_options.keys())),
    description='Mercado:',
    disabled=False,
)

widgets.interact(mostrar_mercado, mercado=selector)

## 12. Normalización de Nombres Homónimos y Control de Integridad Referencial

En esta sección se implementa un diccionario de mapeo para estandarizar las cadenas de texto de los equipos entre dos fuentes de datos heterogéneas (*Football-Data* y *Understat*). Posteriormente, se ejecuta una prueba de control mediante diferencia de conjuntos ($A \setminus B$) para validar que la intersección de las entidades sea exacta, garantizando una sincronización del 100% que evite errores lógicos de desalineación o valores nulos (`NaN`) al cruzar las cuotas con las predicciones del modelo.

In [ ]:
# Definimos el puente de nombres
mapeo_nombres = {
    'Dortmund': 'Borussia Dortmund',
    'Ein Frankfurt': 'Eintracht Frankfurt',
    'FC Koln': 'FC Cologne',
    'Hamburg': 'Hamburger SV',
    'Heidenheim': 'FC Heidenheim',
    'Leverkusen': 'Bayer Leverkusen',
    'M\'gladbach': 'Borussia M.Gladbach',
    'Mainz': 'Mainz 05',
    'RB Leipzig': 'RasenBallsport Leipzig',
    'St Pauli': 'St. Pauli',
    'Stuttgart': 'VfB Stuttgart'
}

# Aplicamos la transformación a las columnas del dataset de cuotas
df_bundesliga['HomeTeam'] = df_bundesliga['HomeTeam'].replace(mapeo_nombres)
df_bundesliga['AwayTeam'] = df_bundesliga['AwayTeam'].replace(mapeo_nombres)

# --- VERIFICACIÓN DE SEGURIDAD ---
# Volvemos a extraer los equipos de la temporada de cuotas corregida
equipos_cuotas_corregidos = set(df_bundesliga['HomeTeam'].unique())
equipos_modelo = set(teams['team'].unique()) # Tu tabla de mapeo ID-Nombre del modelo

faltantes = equipos_cuotas_corregidos - equipos_modelo

print(f"Equipos de la temporada 2025 que el modelo NO reconoce: {len(faltantes)}")
if len(faltantes) > 0:
    print(f" Alerta, aún faltan por corregir: {faltantes}")
else:
    print(" ¡Sincronización perfecta! El 100% de los equipos de la temporada 2025 ya son reconocidos por el modelo.")

## 13. Algoritmo de Backtesting de Estrategias Competitivas (Profesor vs. Modelo)

En esta sección se implementa el motor de simulación histórica para contrastar dos filosofías de inversión opuestas: la **Estrategia del Profesor**, que representa la eficiencia del mercado al apostar sistemáticamente por las cuotas implícitas más bajas (el favorito), y la **Estrategia Bayesiana**, que ejecuta operaciones únicamente en escenarios donde se detecta **Valor Esperado Positivo ($EV+$)** mediante la ecuación:

$$EV = (P_{\text{Modelo}} \times \text{Cuota}_{\text{Casa}}) - 1$$

### Lógica Conceptual del Algoritmo:
* **Normalización del Mercado:** Para calcular el $RPS$ del Profesor, las probabilidades implícitas brutas ($1/\text{Cuota}$) se dividen por su suma total, neutralizando el *Overround* (margen comercial de la casa) para aislar la verdadera distribución probabilística del mercado.
* **Filtro Operacional:** El Profesor opera bajo un umbral restrictivo ($\text{Cuota} < 4.0$) para evitar eventos de alta volatilidad, mientras que el modelo bayesiano actúa como un buscador de ineficiencias, gatillando una apuesta solo si el $EV > 0$ en el mercado 1X2 y seleccionando el vector de máxima ventaja matemática ($\arg\max(EV)$).

In [ ]:
import numpy as np
import pandas as pd

# Diccionario para mapear resultados a índices (0: Local, 1: Empate, 2: Visitante)
result_map = {'H': 0, 'D': 1, 'A': 2}

def evaluate_strategies(df, trace_model, teams_df):
    # --- Contadores: Estrategia Profesor ---
    prof_correct = 0
    prof_bets = 0
    prof_rps_scores = []

    # --- Contadores: Tu Estrategia (EV > 0.1) ---
    mod_correct = 0
    mod_bets = 0
    mod_rps_scores = []

    for idx, row in df.iterrows():
        # 1. Validaciones iniciales
        if pd.isna(row['B365H']) or pd.isna(row['B365D']) or pd.isna(row['B365A']):
            continue

        real_result = row['FTR']
        if real_result not in result_map:
            continue

        outcome_idx = result_map[real_result]
        odds = np.array([row['B365H'], row['B365D'], row['B365A']])

        # ANÁLISIS ESTRATEGIA 1: EL PROFESOR
        # Condición: Todas las cuotas deben ser menores a 4
        if np.all(odds < 4.0):
            prof_bets += 1
            bet_idx = np.argmin(odds) # Apuesta al favorito (menor cuota)
            if bet_idx == outcome_idx:
                prof_correct += 1

        # RPS del mercado (Probabilidades implícitas ajustadas por Overround)
        implied_probs = 1 / odds
        market_probs = implied_probs / np.sum(implied_probs) # Normalización estricta a 1.0
        prof_rps_scores.append(rps(market_probs, outcome_idx))

        # ANÁLISIS ESTRATEGIA 2: MODELO BAYESIANO
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]
        except IndexError:
            # Si un equipo recién ascendido falta en tu matriz de Understat, lo omitimos
            continue

        # Generar expectativas y probabilidades usando tus funciones previas
        h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)
        model_probs = np.array(win_draw_loss(h_exp, a_exp))

        # Calcular RPS del modelo
        mod_rps_scores.append(rps(model_probs, outcome_idx))

        # Calcular Valor Esperado (EV) para 1, X, 2
        evs = (model_probs * odds) - 1

        # ¡CAMBIO AQUÍ!: Solo apostar si el EV supera el 0.1 (10% de ventaja mínima)
        if np.any(evs > 0.1):
            mod_bets += 1
            best_bet_idx = np.argmax(evs) # Tomar la apuesta de mayor valor (EV máximo)

            if best_bet_idx == outcome_idx:
                mod_correct += 1
    # CÁLCULO DE MÉTRICAS FINALES
    prof_accuracy = (prof_correct / prof_bets) if prof_bets > 0 else 0
    mod_accuracy = (mod_correct / mod_bets) if mod_bets > 0 else 0

    prof_avg_rps = np.mean(prof_rps_scores)
    mod_avg_rps = np.mean(mod_rps_scores)

    # Imprimir Reporte
    print("=" * 50)
    print(" RESULTADOS DEL DUELO DE ESTRATEGIAS (BUNDESLIGA)")
    print("=" * 50)
    print("\n[1] ESTRATEGIA DEL PROFESOR (Cuotas < 4)")
    print(f"Apuestas realizadas: {prof_bets}")
    print(f"Precisión (Accuracy): {prof_accuracy:.2%}")
    print(f"RPS Promedio: {prof_avg_rps:.4f} (Menor es mejor)")

    print("\n[2] TU ESTRATEGIA (Modelo Bayesiano EV > 10%)")
    print(f"Apuestas realizadas: {mod_bets}")
    print(f"Precisión (Accuracy): {mod_accuracy:.2%}")
    print(f"RPS Promedio: {mod_avg_rps:.4f} (Menor es mejor)")
    print("=" * 50)

# Ejecutar el análisis
evaluate_strategies(df_bundesliga, trace, teams)

## 14. Backtesting Financiero con Filtro de Umbral de Rentabilidad (EV+ Mínimo)

En esta sección se evoluciona el motor de backtesting incorporando un parámetro de control de riesgo (`ev_threshold`) que actúa como un filtro de ruido estadístico, restringiendo las operaciones del modelo bayesiano únicamente a escenarios de alta convicción donde el Valor Esperado Positivo supera un umbral mínimo preestablecido ($10\%$). El algoritmo rastrea la trayectoria cronológica de los fondos (*banca_modelo* vs. *banca_profesor*) aplicando una gestión de riesgo de stake fijo (*Flat Stake* de $\$10$ USD) para aislar el impacto real de las decisiones matemáticas frente al desgaste transaccional provocado por las casas de apuestas.

In [ ]:
def ejecutar_backtesting_profesional(df, trace_model, teams_df, capital_inicial=1000.0, ev_threshold=0.1):
    """
    ev_threshold: Porcentaje mínimo de ventaja requerido (0.1 = 10% de EV+ mínimo)
    """
    result_map = {'H': 0, 'D': 1, 'A': 2}

    banca_profesor = [capital_inicial]
    banca_modelo = [capital_inicial]

    cap_prof = capital_inicial
    cap_mod = capital_inicial

    ops_prof = 0
    ops_mod = 0

    for idx, row in df.iterrows():
        if pd.isna(row['B365H']) or pd.isna(row['B365D']) or pd.isna(row['B365A']):
            continue

        real_result = row['FTR']
        if real_result not in result_map:
            continue

        outcome_idx = result_map[real_result]
        odds = np.array([row['B365H'], row['B365D'], row['B365A']])

        # ESTRATEGIA PROFESOR (Se mantiene idéntica para comparar)
        if np.all(odds < 4.0):
            stake_p = 10.0
            if cap_prof >= stake_p:
                ops_prof += 1
                bet_idx_prof = np.argmin(odds)
                cap_prof -= stake_p
                if bet_idx_prof == outcome_idx:
                    cap_prof += stake_p * odds[bet_idx_prof]
        banca_profesor.append(cap_prof)

        # ESTRATEGIA EV+ CON FILTRO DE EXCELENCIA
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)
            model_probs = np.array(win_draw_loss(h_exp, a_exp))

            evs = (model_probs * odds) - 1

            if np.any(evs > ev_threshold):
                stake_m = 10.0
                if cap_mod >= stake_m:
                    ops_mod += 1
                    best_bet_idx_mod = np.argmax(evs)

                    cap_mod -= stake_m
                    if best_bet_idx_mod == outcome_idx:
                        cap_mod += stake_m * odds[best_bet_idx_mod]

        except IndexError:
            pass

        banca_modelo.append(cap_mod)

    # REPORTE
    print("=" * 60)
    print(f"   BACKTESTING OPTIMIZADO (UMBRAL EV+: {ev_threshold:.0%})")
    print("=" * 60)
    print(f"[Profesor]  Apuestas: {ops_prof:3d} | Capital Final: ${cap_prof:.2f} USD")
    print(f"[ESTRATEGIA EV+] Apuestas: {ops_mod:3d} | Capital Final: ${cap_mod:.2f} USD")
    print("=" * 60)

    plt.figure(figsize=(12, 6))
    plt.plot(banca_profesor, label=f'Profesor ({ops_prof} unds)', color='crimson', alpha=0.8)
    plt.plot(banca_modelo, label=f'ESTRATEGIA EV+ Filtrado ({ops_mod} unds)', color='teal', lw=2.5)
    plt.axhline(y=capital_inicial, color='black', linestyle='--', alpha=0.5)
    plt.title(f'Efecto del Filtro de Ruido en el Modelo (EV > {ev_threshold:.0%})')
    plt.xlabel('Partidos Cronológicos')
    plt.ylabel('Banca (USD)')
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()

ejecutar_backtesting_profesional(df_bundesliga, trace, teams, ev_threshold=0.05)


## 14. Motor de Backtesting Financiero en el Mercado de Goles (Over/Under 2.5)

En esta sección se implementa el algoritmo de validación comercial definitivo para el mercado de totales (Over/Under 2.5 goles), explotando la ventaja matemática natural del modelo de Poisson para estimar sumas acumuladas de goles. El simulador contrasta cronológicamente la estrategia del mercado (apostar a la cuota menor u opción favorita) frente a las señales de **Valor Esperado Positivo ($EV+$)** gatilladas por el modelo probabilístico, aplicando una gestión de riesgo con un porcentaje de apuesta constante (*Flat Stake* de $\$10$ USD) para medir la viabilidad financiera real de la estrategia de goles.

In [ ]:
def ejecutar_backtesting_over_under(df, trace_model, teams_df, capital_inicial=1000.0, ev_threshold=0.02):
    """
    Backtesting estricto para el mercado Over/Under 2.5 goles.
    ev_threshold: Margen mínimo de EV+ (0.02 = 2% de ventaja mínima).
    """
    if 'B365>2.5' not in df.columns or 'B365<2.5' not in df.columns:
        print("Error: No se encontraron las columnas 'B365>2.5' y 'B365<2.5' en df_bundesliga.")
        return

    banca_profesor = [capital_inicial]
    banca_modelo = [capital_inicial]

    cap_prof = capital_inicial
    cap_mod = capital_inicial

    ops_prof = 0
    ops_mod = 0
    stake_fijo = 10.0

    for idx, row in df.iterrows():
        if pd.isna(row['B365>2.5']) or pd.isna(row['B365<2.5']) or pd.isna(row['FTHG']) or pd.isna(row['FTAG']):
            continue

        goles_totales = int(row['FTHG'] + row['FTAG'])
        resultado_real = 'over' if goles_totales > 2.5 else 'under'

        odd_over = float(row['B365>2.5'])
        odd_under = float(row['B365<2.5'])
        cuotas_ou = np.array([odd_over, odd_under])

        # ESTRATEGIA DEL PROFESOR (Favorito del Mercado de Goles)
        if cap_prof >= stake_fijo:
            ops_prof += 1
            idx_menor = np.argmin(cuotas_ou)
            pred_prof = 'over' if idx_menor == 0 else 'under'
            cuota_prof = cuotas_ou[idx_menor]

            cap_prof -= stake_fijo
            if pred_prof == resultado_real:
                cap_prof += stake_fijo * cuota_prof
        banca_profesor.append(cap_prof)

        # ESTRATEGIA EV+: MODELO BAYESIANO POISSON (EV+)
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)

            prob_ou_modelo = over_under_goals(h_exp, a_exp, 2.5)
            p_over = prob_ou_modelo['over']
            p_under = prob_ou_modelo['under']

            ev_over = (p_over * odd_over) - 1
            ev_under = (p_under * odd_under) - 1

            evs = np.array([ev_over, ev_under])
            best_idx = np.argmax(evs)
            max_ev = evs[best_idx]

            if max_ev > ev_threshold:
                if cap_mod >= stake_fijo:
                    ops_mod += 1
                    pred_mod = 'over' if best_idx == 0 else 'under'
                    cuota_mod = cuotas_ou[best_idx]

                    cap_mod -= stake_fijo
                    if pred_mod == resultado_real:
                        cap_mod += stake_fijo * cuota_mod

        except (IndexError, KeyError):
            pass

        banca_modelo.append(cap_mod)

    # REPORTE DE RESULTADOS
    print("=" * 70)
    print("   BACKTESTING DE COMPROBACIÓN: MERCADO OVER/UNDER 2.5 GOLES")
    print("=" * 70)
    print(f"[Profesor O/U] Apuestas: {ops_prof:3d} | Capital Final: ${cap_prof:.2f} USD")
    print(f"[ESTRATEGIA EV+ O/U] Apuestas: {ops_mod:3d} | Capital Final: ${cap_mod:.2f} USD")
    print("=" * 70)

    plt.figure(figsize=(12, 6))
    plt.plot(banca_profesor, label=f'Profesor O/U Favorito ({ops_prof} apuestas)', color='crimson', alpha=0.6, lw=2)
    plt.plot(banca_modelo, label=f'ESTRATEGIA EV+ O/U ({ops_mod} apuestas)', color='darkorange', lw=2.5)
    plt.axhline(y=capital_inicial, color='black', linestyle='--', alpha=0.5)
    plt.title('Comparativa de Rendimiento Financiero: Mercado de Goles Totales (O/U 2.5)', fontsize=13, fontweight='bold')
    plt.xlabel('Partidos Procesados de la Temporada')
    plt.ylabel('Banca (USD)')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

ejecutar_backtesting_over_under(df_bundesliga, trace, teams, ev_threshold=0.05)


## 15. Motor de Backtesting Financiero en el Mercado de Hándicap Asiático

En esta sección se implementa la lógica de liquidación y simulación comercial para el mercado de Hándicap Asiático. El algoritmo introduce la función `evaluar_retorno_ah` para calcular con precisión los cinco escenarios de retorno de este mercado fraccionado (ganada, mitad ganada, *push* o nula, mitad perdida y perdida completa). Posteriormente, el motor de *backtesting* contrasta la estrategia del mercado frente a las desviaciones de cuotas identificadas por el modelo probabilístico, ejecutando transacciones bajo un esquema de capital constante (*Flat Stake* de $\$10$ USD) para medir la viabilidad financiera de las ventajas estimadas en líneas de ventaja y desventaja de goles.

In [ ]:
def evaluar_retorno_ah(goles_local, goles_visitante, linea_home, tipo_apuesta, cuota, stake):
    """
    Calcula matemáticamente el retorno de una apuesta de Hándicap Asiático.
    linea_home: Hándicap de cierre para el Local (columna AHCh de football-data)
    tipo_apuesta: 'home' o 'away'
    """
    diff_goles = goles_local - goles_visitante

    v_score = (diff_goles + linea_home) if tipo_apuesta == 'home' else -(diff_goles + linea_home)

    if v_score > 0.25:
        return stake * cuota
    elif v_score == 0.25:
        return stake * (1 + (cuota - 1)/2)
    elif v_score == 0:
        return stake
    elif v_score == -0.25:
        return stake * 0.5
    else:
        return 0.0

def ejecutar_backtesting_handicap(df, trace_model, teams_df, capital_inicial=1000.0, ev_threshold=0.03):
    if 'AHCh' not in df.columns or 'B365AHH' not in df.columns or 'B365AHA' not in df.columns:
        print("Error: No se encontraron las columnas de Hándicap Asiático ('AHCh', 'B365AHH', 'B365AHA').")
        return

    banca_profesor = [capital_inicial]
    banca_modelo = [capital_inicial]
    cap_prof, cap_mod = capital_inicial, capital_inicial
    ops_prof, ops_mod = 0, 0
    stake_fijo = 10.0

    for idx, row in df.iterrows():
        if pd.isna(row['AHCh']) or pd.isna(row['B365AHH']) or pd.isna(row['B365AHA']) or pd.isna(row['FTHG']):
            continue

        gl, gv = int(row['FTHG']), int(row['FTAG'])
        linea_actual = float(row['AHCh'])
        odd_m_home = float(row['B365AHH'])
        odd_m_away = float(row['B365AHA'])
        cuotas_mercado = np.array([odd_m_home, odd_m_away])

        # ESTRATEGIA DEL PROFESOR (El favorito de la línea)
        if cap_prof >= stake_fijo:
            ops_prof += 1
            idx_menor = np.argmin(cuotas_mercado)
            tipo_p = 'home' if idx_menor == 0 else 'away'
            cuota_p = cuotas_mercado[idx_menor]

            cap_prof -= stake_fijo
            cap_prof += evaluar_retorno_ah(gl, gv, linea_actual, tipo_p, cuota_p, stake_fijo)
        banca_profesor.append(cap_prof)

        # ESTRATEGIA EV+: MODELO BAYESIANO POISSON (EV+ AH)
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)

            ah_table_modelo = asian_handicap_table(h_exp, a_exp)

            linea_encontrada = None
            for k in ah_table_modelo.keys():
                if abs(k - linea_actual) < 1e-4:
                    linea_encontrada = k
                    break

            if linea_encontrada is not None:
                odd_mod_home = ah_table_modelo[linea_encontrada]['home_odds']
                odd_mod_away = ah_table_modelo[linea_encontrada]['away_odds']

                ev_home = (odd_m_home / odd_mod_home) - 1
                ev_away = (odd_m_away / odd_mod_away) - 1

                evs = np.array([ev_home, ev_away])
                best_idx = np.argmax(evs)
                max_ev = evs[best_idx]

                if max_ev > ev_threshold:
                    if cap_mod >= stake_fijo:
                        ops_mod += 1
                        tipo_m = 'home' if best_idx == 0 else 'away'
                        cuota_m = cuotas_mercado[best_idx]

                        cap_mod -= stake_fijo
                        cap_mod += evaluar_retorno_ah(gl, gv, linea_actual, tipo_m, cuota_m, stake_fijo)

        except (IndexError, KeyError):
            pass
        banca_modelo.append(cap_mod)

    # DESPLIEGUE
    print("=" * 70)
    print("   BACKTESTING DE COMPROBACIÓN: MERCADO HÁNDICAP ASIÁTICO")
    print("=" * 70)
    print(f"[Profesor AH] Apuestas: {ops_prof:3d} | Capital Final: ${cap_prof:.2f} USD")
    print(f"[ESTRATEGIA EV+ AH] Apuestas: {ops_mod:3d} | Capital Final: ${cap_mod:.2f} USD")
    print("=" * 70)

    plt.figure(figsize=(12, 6))
    plt.plot(banca_profesor, label=f'Profesor AH ({ops_prof} apuestas)', color='crimson', alpha=0.6, lw=2)
    plt.plot(banca_modelo, label=f'ESTRATEGIA EV+ AH ({ops_mod} apuestas)', color='royalblue', lw=2.5)
    plt.axhline(y=capital_inicial, color='black', linestyle='--', alpha=0.5)
    plt.title('Comparativa de Rendimiento Financiero: Hándicap Asiático', fontsize=13, fontweight='bold')
    plt.xlabel('Partidos Procesados')
    plt.ylabel('Banca (USD)')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

ejecutar_backtesting_handicap(df_bundesliga, trace, teams, ev_threshold=0.05)


## 16. Backtesting Financiero con Ventana de Seguridad y Control de Sobreconfianza (EV+ Acotado)

En esta sección se diseña una variante de backtesting para el mercado 1X2 introduciendo un filtro de cota superior e inferior (`ev_min <= max_ev <= ev_max`). Esta lógica conceptual asume que las desviaciones masivas de Valor Esperado ($EV > 15\%-20\%$) calculadas por modelos imperfectos no representan verdaderas ineficiencias del mercado, sino "trampas de sobreconfianza" o fallos de calibración matemática generados al ignorar variables cualitativas dinámicas, por lo que acotar la operatividad a una ventana de valor seguro mitiga el riesgo sistémico de *overfitting* sin abandonar el control estricto de capital (*Flat Stake* de $\$10$ USD).

In [ ]:
def ejecutar_backtesting_ventana_ev(df, trace_model, teams_df, capital_inicial=1000.0, ev_min=0.05, ev_max=0.20):
    """
    Backtesting 1X2 utilizando el modelo original pero limitando las apuestas
    a una ventana de Valor Esperado para evitar trampas de sobreconfianza.
    """
    result_map = {'H': 0, 'D': 1, 'A': 2}

    banca_profesor = [capital_inicial]
    banca_modelo = [capital_inicial]

    cap_prof = capital_inicial
    cap_mod = capital_inicial

    ops_prof = 0
    ops_mod = 0
    stake_fijo = 10.0

    for idx, row in df.iterrows():
        if pd.isna(row['B365H']) or pd.isna(row['B365D']) or pd.isna(row['B365A']):
            continue

        real_result = row['FTR']
        if real_result not in result_map:
            continue

        outcome_idx = result_map[real_result]
        odds = np.array([row['B365H'], row['B365D'], row['B365A']])

        # ESTRATEGIA DEL PROFESOR (Regla original de 1X2)
        if np.all(odds < 4.0):
            if cap_prof >= stake_fijo:
                ops_prof += 1
                bet_idx_prof = np.argmin(odds)

                cap_prof -= stake_fijo
                if bet_idx_prof == outcome_idx:
                    cap_prof += stake_fijo * odds[bet_idx_prof]
        banca_profesor.append(cap_prof)

        # ESTRATEGIA EV+: MODELO PURO CON VENTANA DE SEGURIDAD
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)
            model_probs = np.array(win_draw_loss(h_exp, a_exp))

            evs = (model_probs * odds) - 1
            best_bet_idx_mod = np.argmax(evs)
            max_ev = evs[best_bet_idx_mod]

            if ev_min <= max_ev <= ev_max:
                if cap_mod >= stake_fijo:
                    ops_mod += 1

                    cap_mod -= stake_fijo
                    if best_bet_idx_mod == outcome_idx:
                        cap_mod += stake_fijo * odds[best_bet_idx_mod]

        except IndexError:
            pass

        banca_modelo.append(cap_mod)

    # REPORTE EN CONSOLA
    print("=" * 70)
    print(f"   BACKTESTING 1X2 - VENTANA DE VALOR SEGURO ({ev_min:.0%} a {ev_max:.0%})")
    print("=" * 70)
    print(f"[Profesor]  Apuestas: {ops_prof:3d} | Capital Final: ${cap_prof:.2f} USD")
    print(f"[ESTRATEGIA EV+] Apuestas: {ops_mod:3d} | Capital Final: ${cap_mod:.2f} USD")
    print("=" * 70)

    plt.figure(figsize=(12, 6))
    plt.plot(banca_profesor, label=f'Profesor ({ops_prof} apuestas)', color='crimson', alpha=0.5, lw=2)
    plt.plot(banca_modelo, label=f'ESTRATEGIA EV+ Acotado ({ops_mod} apuestas)', color='teal', lw=2.5)
    plt.axhline(y=capital_inicial, color='black', linestyle='--', alpha=0.5)

    plt.title(f'Filtro de Overfitting: Operando solo cuando {ev_min:.0%} <= EV <= {ev_max:.0%}', fontsize=13, fontweight='bold')
    plt.xlabel('Partidos de la Temporada')
    plt.ylabel('Banca (USD)')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

ejecutar_backtesting_ventana_ev(df_bundesliga, trace, teams, ev_min=0.05, ev_max=0.15)


## 17. Backtesting de Goles (Over/Under 2.5) con Ventana de Seguridad y Control de Anomalías Estadísticas

En esta sección se adapta el concepto de ventana de seguridad (`ev_min <= max_ev <= ev_max`) específicamente al mercado de goles (Over/Under 2.5). Al cruzar las probabilidades de la distribución conjunta de Poisson con las cuotas comerciales, el algoritmo descarta los extremos: ignora ventajas marginales propensas al ruido y bloquea señales con un Valor Esperado excesivamente alto que delaten fallos de calibración en partidos atípicos, optimizando la consistencia financiera bajo un esquema de capital constante (*Flat Stake* del $1\%$).

In [ ]:
def ejecutar_backtesting_ou_ventana(df, trace_model, teams_df, capital_inicial=1000.0, ev_min=0.05, ev_max=0.20):
    """
    Backtesting para el mercado Over/Under 2.5 goles acotando el EV+
    para evitar anomalías estadísticas y sobreconfianza.
    """
    if 'B365>2.5' not in df.columns or 'B365<2.5' not in df.columns:
        print("Error: No se encontraron las columnas de cuotas Over/Under de Bet365.")
        return

    banca_profesor = [capital_inicial]
    banca_modelo = [capital_inicial]

    cap_prof = capital_inicial
    cap_mod = capital_inicial

    ops_prof = 0
    ops_mod = 0
    stake_fijo = 10.0

    for idx, row in df.iterrows():
        if pd.isna(row['B365>2.5']) or pd.isna(row['B365<2.5']) or pd.isna(row['FTHG']):
            continue

        goles_totales = int(row['FTHG'] + row['FTAG'])
        resultado_real = 'over' if goles_totales > 2.5 else 'under'

        odd_over = float(row['B365>2.5'])
        odd_under = float(row['B365<2.5'])
        cuotas_ou = np.array([odd_over, odd_under])

        # ESTRATEGIA DEL PROFESOR (Favorito del mercado O/U)
        if cap_prof >= stake_fijo:
            ops_prof += 1
            idx_menor = np.argmin(cuotas_ou)
            pred_prof = 'over' if idx_menor == 0 else 'under'
            cuota_prof = cuotas_ou[idx_menor]

            cap_prof -= stake_fijo
            if pred_prof == resultado_real:
                cap_prof += stake_fijo * cuota_prof
        banca_profesor.append(cap_prof)

        # ESTRATEGIA EV+: POISSON CON VENTANA DE SEGURIDAD (5% - 20%)
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)

            prob_ou_modelo = over_under_goals(h_exp, a_exp, 2.5)
            p_over = prob_ou_modelo['over']
            p_under = prob_ou_modelo['under']

            ev_over = (p_over * odd_over) - 1
            ev_under = (p_under * odd_under) - 1

            evs = np.array([ev_over, ev_under])
            best_idx = np.argmax(evs)
            max_ev = evs[best_idx]

            if ev_min <= max_ev <= ev_max:
                if cap_mod >= stake_fijo:
                    ops_mod += 1
                    pred_mod = 'over' if best_idx == 0 else 'under'
                    cuota_mod = cuotas_ou[best_idx]

                    cap_mod -= stake_fijo
                    if pred_mod == resultado_real:
                        cap_mod += stake_fijo * cuota_mod

        except (IndexError, KeyError):
            pass

        banca_modelo.append(cap_mod)

    # REPORTE FINAL
    print("=" * 70)
    print(f"   BACKTESTING OVER/UNDER 2.5 - VENTANA DE VALOR ({ev_min:.0%} a {ev_max:.0%})")
    print("=" * 70)
    print(f"[Profesor O/U]  Apuestas: {ops_prof:3d} | Capital Final: ${cap_prof:.2f} USD")
    print(f"[ESTRATEGIA EV+ O/U] Apuestas: {ops_mod:3d} | Capital Final: ${cap_mod:.2f} USD")
    print("=" * 70)

    plt.figure(figsize=(12, 6))
    plt.plot(banca_profesor, label=f'Profesor O/U ({ops_prof} apuestas)', color='crimson', alpha=0.5, lw=2)
    plt.plot(banca_modelo, label=f'ESTRATEGIA EV+ Acotado ({ops_mod} apuestas)', color='darkorange', lw=2.5)
    plt.axhline(y=capital_inicial, color='black', linestyle='--', alpha=0.5)

    plt.title(f'Análisis de Goles O/U 2.5 - Ventana de Seguridad ({ev_min:.0%} <= EV <= {ev_max:.0%})', fontsize=13, fontweight='bold')
    plt.xlabel('Partidos de la Temporada')
    plt.ylabel('Banca (USD)')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

ejecutar_backtesting_ou_ventana(df_bundesliga, trace, teams, ev_min=0.05, ev_max=0.15)


## 18. Backtesting de Hándicap Asiático con Ventana de Seguridad y Control de Anomalías de Cuota

En esta sección se integra la función matemática de retornos fraccionados `evaluar_retorno_ah` dentro de una arquitectura de simulación acotada por una ventana de valor seguro (`ev_min <= max_ev <= ev_max`). Al mapear las líneas dinámicas de hándicap impuestas por el mercado contra la tabla de probabilidades calculada por el modelo de Poisson, el algoritmo descarta ineficiencias marginales y mitiga los fallos de calibración en cuotas atípicas del mercado asiático, operando bajo un esquema de stake constante del $1\%$ para blindar la estabilidad financiera de la banca.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# EVALUADOR DE PAGOS DE HÁNDICAP ASIÁTICO
def evaluar_retorno_ah(goles_local, goles_visitante, linea_home, tipo_apuesta, cuota, stake):
    diff_goles = goles_local - goles_visitante

    v_score = (diff_goles + linea_home) if tipo_apuesta == 'home' else -(diff_goles + linea_home)

    if v_score > 0.25:
        return stake * cuota
    elif v_score == 0.25:
        return stake * (1 + (cuota - 1)/2)
    elif v_score == 0:
        return stake
    elif v_score == -0.25:
        return stake * 0.5
    else:
        return 0.0

# MOTOR DE BACKTESTING CON VENTANA ACOTA
def ejecutar_backtesting_ah_ventana(df, trace_model, teams_df, capital_inicial=1000.0, ev_min=0.05, ev_max=0.20):
    if 'AHCh' not in df.columns or 'B365AHH' not in df.columns or 'B365AHA' not in df.columns:
        print("Error: No se encontraron las columnas de Hándicap Asiático en el DataFrame.")
        return

    banca_profesor = [capital_inicial]
    banca_modelo = [capital_inicial]

    cap_prof = capital_inicial
    cap_mod = capital_inicial

    ops_prof = 0
    ops_mod = 0
    stake_fijo = 10.0

    for idx, row in df.iterrows():
        if pd.isna(row['AHCh']) or pd.isna(row['B365AHH']) or pd.isna(row['B365AHA']) or pd.isna(row['FTHG']):
            continue

        gl, gv = int(row['FTHG']), int(row['FTAG'])
        linea_actual = float(row['AHCh'])
        odd_m_home = float(row['B365AHH'])
        odd_m_away = float(row['B365AHA'])
        cuotas_mercado = np.array([odd_m_home, odd_m_away])

        # ESTRATEGIA DEL PROFESOR (Favorito de la Línea de Hándicap)
        if cap_prof >= stake_fijo:
            ops_prof += 1
            idx_menor = np.argmin(cuotas_mercado)
            tipo_p = 'home' if idx_menor == 0 else 'away'
            cuota_p = cuotas_mercado[idx_menor]

            cap_prof -= stake_fijo
            cap_prof += evaluar_retorno_ah(gl, gv, linea_actual, tipo_p, cuota_p, stake_fijo)
        banca_profesor.append(cap_prof)

        # ESTRATEGIA EV+: WINDOW FILTER (5% <= EV <= 20%)
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']

        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)
            ah_table_modelo = asian_handicap_table(h_exp, a_exp)

            linea_encontrada = None
            for k in ah_table_modelo.keys():
                if abs(k - linea_actual) < 1e-4:
                    linea_encontrada = k
                    break

            if linea_encontrada is not None:
                odd_mod_home = ah_table_modelo[linea_encontrada]['home_odds']
                odd_mod_away = ah_table_modelo[linea_encontrada]['away_odds']

                ev_home = (odd_m_home / odd_mod_home) - 1
                ev_away = (odd_m_away / odd_mod_away) - 1

                evs = np.array([ev_home, ev_away])
                best_idx = np.argmax(evs)
                max_ev = evs[best_idx]

                if ev_min <= max_ev <= ev_max:
                    if cap_mod >= stake_fijo:
                        ops_mod += 1
                        tipo_m = 'home' if best_idx == 0 else 'away'
                        cuota_m = cuotas_mercado[best_idx]

                        cap_mod -= stake_fijo
                        cap_mod += evaluar_retorno_ah(gl, gv, linea_actual, tipo_m, cuota_m, stake_fijo)

        except (IndexError, KeyError):
            pass

        banca_modelo.append(cap_mod)

    # REPORTE DE RENDIMIENTO
    print("=" * 70)
    print(f"   BACKTESTING HÁNDICAP ASIÁTICO - VENTANA DE VALOR ({ev_min:.0%} a {ev_max:.0%})")
    print("=" * 70)
    print(f"[Profesor AH]  Apuestas: {ops_prof:3d} | Capital Final: ${cap_prof:.2f} USD")
    print(f"[ESTRATEGIA EV+ AH] Apuestas: {ops_mod:3d} | Capital Final: ${cap_mod:.2f} USD")
    print("=" * 70)

    plt.figure(figsize=(12, 6))
    plt.plot(banca_profesor, label=f'Profesor AH ({ops_prof} apuestas)', color='crimson', alpha=0.5, lw=2)
    plt.plot(banca_modelo, label=f'ESTRATEGIA EV+ AH ({ops_mod} apuestas)', color='royalblue', lw=2.5)
    plt.axhline(y=capital_inicial, color='black', linestyle='--', alpha=0.5)

    plt.title(f'Hándicap Asiático - Ventana de Seguridad ({ev_min:.0%} <= EV <= {ev_max:.0%})', fontsize=13, fontweight='bold')
    plt.xlabel('Partidos de la Temporada')
    plt.ylabel('Banca (USD)')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()

ejecutar_backtesting_ah_ventana(df_bundesliga, trace, teams, ev_min=0.05, ev_max=0.15)


## 19. Simulación Concurrida de Portafolio Multi-Mercado con Ventana de Seguridad Consolidada

En esta sección se integra el entorno algorítmico definitivo mediante una función de simulación paralela que evalúa simultáneamente el comportamiento financiero del modelo en tres dimensiones de liquidez: el mercado principal de resultados (1X2), la línea de totales (Over/Under 2.5) y las posiciones de Hándicap Asiático. El sistema procesa de forma síncrona cada evento aplicando la restricción de ventana simétrica ($5\% \le EV \le 15\%$) y una política unificada de asignación de capital constante (*Flat Stake* del $1\%$), proyectando las trayectorias patrimoniales cruzadas en un único lienzo visual para auditar las ventajas operativas consolidadas del modelo frente a la fricción comercial.

In [ ]:
def evaluar_retorno_ah(goles_local, goles_visitante, linea_home, tipo_apuesta, cuota, stake):
    diff_goles = goles_local - goles_visitante
    v_score = (diff_goles + linea_home) if tipo_apuesta == 'home' else -(diff_goles + linea_home)

    if v_score > 0.25:     return stake * cuota
    elif v_score == 0.25:   return stake * (1 + (cuota - 1)/2)
    elif v_score == 0:      return stake
    elif v_score == -0.25:  return stake * 0.5
    else:                   return 0.0

def simular_portafolio_completo(df, trace_model, teams_df, capital_inicial=1000.0, ev_min=0.05, ev_max=0.15):
    result_map = {'H': 0, 'D': 1, 'A': 2}
    stake_fijo = 10.0 # 1% fijo

    # Inicializar balances individuales para las 6 líneas
    cap_prof_1x2, cap_mod_1x2 = capital_inicial, capital_inicial
    cap_prof_ou,  cap_mod_ou  = capital_inicial, capital_inicial
    cap_prof_ah,  cap_mod_ah  = capital_inicial, capital_inicial

    # Inicializar contadores de apuestas
    ops_prof_1x2, ops_mod_1x2 = 0, 0
    ops_prof_ou, ops_mod_ou = 0, 0
    ops_prof_ah, ops_mod_ah = 0, 0

    # Historiales cronológicos para graficar
    banca_p_1x2, banca_m_1x2 = [capital_inicial], [capital_inicial]
    banca_p_ou,  banca_m_ou  = [capital_inicial], [capital_inicial]
    banca_p_ah,  banca_m_ah  = [capital_inicial], [capital_inicial]

    for idx, row in df.iterrows():
        # Validaciones de datos mínimos del renglón
        if pd.isna(row['FTHG']) or pd.isna(row['FTAG']) or row['FTR'] not in result_map: continue
        if pd.isna(row['B365H']) or pd.isna(row['B365>2.5']) or pd.isna(row['AHCh']): continue

        # Datos reales del partido
        gl, gv = int(row['FTHG']), int(row['FTAG'])
        outcome_idx = result_map[row['FTR']]
        goles_totales = gl + gv
        resultado_real_ou = 'over' if goles_totales > 2.5 else 'under'

        # Cuotas de Mercado
        odds_1x2 = np.array([float(row['B365H']), float(row['B365D']), float(row['B365A'])])
        odds_ou  = np.array([float(row['B365>2.5']), float(row['B365<2.5'])])
        linea_ah = float(row['AHCh'])
        odds_ah  = np.array([float(row['B365AHH']), float(row['B365AHA'])])

        # Identificación de Equipos en tu Modelo
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            # --- CÁLCULOS DEL MODELO BAYESIANO ---
            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)

            # Probabilidades del modelo por mercado
            probs_1x2 = np.array(win_draw_loss(h_exp, a_exp))

            prob_ou_dict = over_under_goals(h_exp, a_exp, 2.5)
            probs_ou = np.array([prob_ou_dict['over'], prob_ou_dict['under']])

            ah_table = asian_handicap_table(h_exp, a_exp)
            linea_key = next((k for k in ah_table.keys() if abs(k - linea_ah) < 1e-4), None)
        except (IndexError, KeyError, StopIteration):
            # Si falla la extracción o el mapeo, se planchan las bancas este partido
            banca_p_1x2.append(cap_prof_1x2); banca_m_1x2.append(cap_mod_1x2)
            banca_p_ou.append(cap_prof_ou);   banca_m_ou.append(cap_mod_ou)
            banca_p_ah.append(cap_prof_ah);   banca_m_ah.append(cap_mod_ah)
            continue

        # EJECUCIÓN PARALELA: MERCADO 1X2
        # Profesor 1X2
        if np.all(odds_1x2 < 4.0) and cap_prof_1x2 >= stake_fijo:
            bet_p = np.argmin(odds_1x2)
            cap_prof_1x2 -= stake_fijo
            if bet_p == outcome_idx: cap_prof_1x2 += stake_fijo * odds_1x2[bet_p]
            ops_prof_1x2 += 1

        # ESTRATEGIA EV+ 1X2 (Con Ventana de Seguridad 5% - 15%)
        evs_1x2 = (probs_1x2 * odds_1x2) - 1
        best_1x2 = np.argmax(evs_1x2)
        if ev_min <= evs_1x2[best_1x2] <= ev_max and cap_mod_1x2 >= stake_fijo:
            cap_mod_1x2 -= stake_fijo
            if best_1x2 == outcome_idx: cap_mod_1x2 += stake_fijo * odds_1x2[best_1x2]
            ops_mod_1x2 += 1

        # EJECUCIÓN PARALELA: MERCADO OVER/UNDER 2.5
        # Profesor O/U (aligned with single-market backtesting behavior)
        if cap_prof_ou >= stake_fijo:
            bet_p = np.argmin(odds_ou)
            pred_p = 'over' if bet_p == 0 else 'under'
            cap_prof_ou -= stake_fijo
            if pred_p == resultado_real_ou: cap_prof_ou += stake_fijo * odds_ou[bet_p]
            ops_prof_ou += 1

        # ESTRATEGIA EV+ O/U (Con Ventana de Seguridad 5% - 15%)
        evs_ou = (probs_ou * odds_ou) - 1
        best_ou = np.argmax(evs_ou)
        if ev_min <= evs_ou[best_ou] <= ev_max and cap_mod_ou >= stake_fijo:
            pred_m = 'over' if best_ou == 0 else 'under'
            cap_mod_ou -= stake_fijo
            if pred_m == resultado_real_ou: cap_mod_ou += stake_fijo * odds_ou[best_ou]
            ops_mod_ou += 1

        # EJECUCIÓN PARALELA: MERCADO HÁNDICAP ASIÁTICO
        # Profesor AH (aligned with single-market backtesting behavior)
        if cap_prof_ah >= stake_fijo:
            bet_p = np.argmin(odds_ah)
            tipo_p = 'home' if bet_p == 0 else 'away'
            cap_prof_ah -= stake_fijo
            cap_prof_ah += evaluar_retorno_ah(gl, gv, linea_ah, tipo_p, odds_ah[bet_p], stake_fijo)
            ops_prof_ah += 1

        # ESTRATEGIA EV+ AH (Con Ventana de Seguridad 5% - 15%)
        if linea_key is not None:
            odds_mod_ah = np.array([ah_table[linea_key]['home_odds'], ah_table[linea_key]['away_odds']])
            evs_ah = (odds_ah / odds_mod_ah) - 1
            best_ah = np.argmax(evs_ah)
            if ev_min <= evs_ah[best_ah] <= ev_max and cap_mod_ah >= stake_fijo:
                tipo_m = 'home' if best_ah == 0 else 'away'
                cap_mod_ah -= stake_fijo
                cap_mod_ah += evaluar_retorno_ah(gl, gv, linea_ah, tipo_m, odds_ah[best_ah], stake_fijo)
                ops_mod_ah += 1

        # Almacenar estados al cierre del partido
        banca_p_1x2.append(cap_prof_1x2); banca_m_1x2.append(cap_mod_1x2)
        banca_p_ou.append(cap_prof_ou);   banca_m_ou.append(cap_mod_ou)
        banca_p_ah.append(cap_prof_ah);   banca_m_ah.append(cap_mod_ah)

    # PROCESAMIENTO VISUAL Y GRAFICACIÓN CONSOLIDADA
    plt.figure(figsize=(14, 8))
    n_partidos = len(banca_p_1x2) - 1

    # Configuración de curvas de visualización
    lineas = {
        '>cuota 1X2':       (banca_p_1x2, 'crimson',    '-', ops_prof_1x2),
        'ESTRATEGIA EV+ 1X2':      (banca_m_1x2, 'crimson',    '-.', ops_mod_1x2),
        '>cuota O/U 2.5':   (banca_p_ou,  'darkorange', '-', ops_prof_ou),
        'ESTRATEGIA EV+ O/U 2.5':  (banca_m_ou,  'darkorange', '-.', ops_mod_ou),
        '>cuota Hándicap':  (banca_p_ah,  'steelblue',  '-', ops_prof_ah),
        'ESTRATEGIA EV+ Hándicap': (banca_m_ah,  'steelblue',  '-.', ops_mod_ah)
    }

    # Dibujar cada línea y estampar su valor final en la gráfica
    for label_base, (datos, color, estilo, num_apuestas) in lineas.items():
        label = f"{label_base} (Apuestas: {num_apuestas})";
        plt.plot(datos, label=label, color=color, linestyle=estilo, lw=2)

        # Estampar el valor de capital final justo en el último nodo de la línea
        val_final = datos[-1]
        plt.text(n_partidos + 1, val_final, f" ${val_final:.2f}",
                 color=color, va='center', fontsize=9, fontweight='bold')

    # Decoración formal del gráfico
    plt.axhline(y=capital_inicial, color='black', linestyle='-.', alpha=0.4, label='Banca Inicial ($1000)')
    plt.title('Simulación de Portafolio Consolidada',
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Partidos Transcurridos en la Bundesliga', fontsize=11)
    plt.ylabel('Capital Disponible (USD)', fontsize=11)
    plt.xlim(0, n_partidos + 25) # Espacio extra a la derecha para que el texto de los precios no se corte
    plt.grid(True, alpha=0.25)
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.show()

    # Despliegue de auditoría en consola
    print("=" * 65)
    print("      RESUMEN DE LIQUIDACIÓN DE PORTAFOLIO (FIN DE TEMPORADA)")
    print("=" * 65)
    print(f"MERCADO 1X2      | >cuota: ${cap_prof_1x2:.2f} USD (Apuestas: {ops_prof_1x2})  | ESTRATEGIA EV+ : ${cap_mod_1x2:.2f} USD (Apuestas: {ops_mod_1x2})")
    print(f"MERCADO O/U 2.5  | >cuota: ${cap_prof_ou:.2f} USD (Apuestas: {ops_prof_ou})  | ESTRATEGIA EV+ : ${cap_mod_ou:.2f} USD (Apuestas: {ops_mod_ou})")
    print(f"MERCADO HÁNDICAP | >cuota: ${cap_prof_ah:.2f} USD (Apuestas: {ops_prof_ah})  | ESTRATEGIA EV+ : ${cap_mod_ah:.2f} USD (Apuestas: {ops_mod_ah})")
    print("=" * 65)

# Ejecutar la simulación definitiva del portafolio acotado al 15%
simular_portafolio_completo(df_bundesliga, trace, teams, ev_min=0.05, ev_max=0.15)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The 'evaluar_retorno_ah' function is already defined earlier in the notebook.
# Including it here again for clarity, but it refers to the same logic.
def evaluar_retorno_ah(goles_local, goles_visitante, linea_home, tipo_apuesta, cuota, stake):
    diff_goles = goles_local - goles_visitante
    v_score = (diff_goles + linea_home) if tipo_apuesta == 'home' else -(diff_goles + linea_home)

    if v_score > 0.25:     return stake * cuota
    elif v_score == 0.25:   return stake * (1 + (cuota - 1)/2)
    elif v_score == 0:      return stake
    elif v_score == -0.25:  return stake * 0.5
    else:                   return 0.0

def simular_banca_unificada_todos_mercados(df, trace_model, teams_df, capital_inicial=1000.0, ev_min=0.05, ev_max=0.15):
    result_map = {'H': 0, 'D': 1, 'A': 2}
    stake_fijo = 10.0 # Stake por cada apuesta en cada mercado

    # Bancas combinadas para ambas estrategias
    banca_profesor_total = [capital_inicial]
    banca_modelo_total = [capital_inicial]

    cap_prof_total = capital_inicial
    cap_mod_total = capital_inicial

    # Contadores de apuestas para resumen
    ops_prof_total = 0
    ops_mod_total = 0

    for idx, row in df.iterrows():
        # Validaciones de datos mínimos para el partido actual
        if pd.isna(row['FTHG']) or pd.isna(row['FTAG']) or row['FTR'] not in result_map:
            banca_profesor_total.append(cap_prof_total)
            banca_modelo_total.append(cap_mod_total)
            continue
        if pd.isna(row['B365H']) or pd.isna(row['B365D']) or pd.isna(row['B365A']):
            banca_profesor_total.append(cap_prof_total)
            banca_modelo_total.append(cap_mod_total)
            continue
        if pd.isna(row['B365>2.5']) or pd.isna(row['B365<2.5']):
            banca_profesor_total.append(cap_prof_total)
            banca_modelo_total.append(cap_mod_total)
            continue
        if pd.isna(row['AHCh']) or pd.isna(row['B365AHH']) or pd.isna(row['B365AHA']):
            banca_profesor_total.append(cap_prof_total)
            banca_modelo_total.append(cap_mod_total)
            continue

        # Datos reales del partido
        gl, gv = int(row['FTHG']), int(row['FTAG'])
        outcome_idx_1x2 = result_map[row['FTR']] # 0:H, 1:D, 2:A
        goles_totales = gl + gv
        outcome_idx_ou = 0 if goles_totales > 2.5 else 1 # 0: Over, 1: Under

        # Cuotas de Mercado (B365 en este caso)
        odds_1x2 = np.array([float(row['B365H']), float(row['B365D']), float(row['B365A'])])
        odds_ou  = np.array([float(row['B365>2.5']), float(row['B365<2.5'])])
        linea_ah = float(row['AHCh'])
        odds_ah  = np.array([float(row['B365AHH']), float(row['B365AHA'])]) # Home AH, Away AH

        # Identificación de Equipos en el Modelo
        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            # --- CÁLCULOS DEL MODELO BAYESIANO PARA OBTENER PROBABILIDADES ----
            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)

            # Probabilidades del modelo por mercado (1X2)
            probs_1x2 = np.array(win_draw_loss(h_exp, a_exp))

            # Probabilidades del modelo por mercado (Over/Under 2.5)
            prob_ou_dict = over_under_goals(h_exp, a_exp, 2.5)
            probs_ou = np.array([prob_ou_dict['over'], prob_ou_dict['under']])

            # Probabilidades/Cuotas del modelo por mercado (Asian Handicap)
            ah_table = asian_handicap_table(h_exp, a_exp)
            linea_key = next((k for k in ah_table.keys() if abs(k - linea_ah) < 1e-4), None)
        except (IndexError, KeyError, StopIteration):
            # Si falla la extracción o el mapeo, las bancas no cambian para este partido
            banca_profesor_total.append(cap_prof_total)
            banca_modelo_total.append(cap_mod_total)
            continue

        # ==================================================================================
        # ESTRATEGIA >CUOTA (COMBINADA): Apuesta al favorito de cada mercado si hay capital
        # ==================================================================================
        bets_placed_prof_in_match = 0
        winnings_prof_in_match = 0

        # >cuota 1X2: Condición del profesor (cuotas < 4.0)
        if np.all(odds_1x2 < 4.0) and cap_prof_total >= stake_fijo:
            bets_placed_prof_in_match += 1
            bet_idx_prof_1x2 = np.argmin(odds_1x2) # Apuesta a la cuota más baja (favorito)
            if bet_idx_prof_1x2 == outcome_idx_1x2:
                winnings_prof_in_match += stake_fijo * odds_1x2[bet_idx_prof_1x2]

        # >cuota O/U 2.5: Siempre apuesta al favorito del mercado si hay capital
        # (Basado en el comportamiento de `ejecutar_backtesting_over_under`)
        if cap_prof_total >= stake_fijo: # Check if enough capital for this specific bet
            bets_placed_prof_in_match += 1
            bet_idx_prof_ou = np.argmin(odds_ou) # Apuesta a la cuota más baja (favorito O/U)
            if bet_idx_prof_ou == outcome_idx_ou:
                winnings_prof_in_match += stake_fijo * odds_ou[bet_idx_prof_ou]

        # >cuota AH: Siempre apuesta al favorito del mercado si hay capital
        # (Basado en el comportamiento de `ejecutar_backtesting_handicap`)
        if cap_prof_total >= stake_fijo: # Check if enough capital for this specific bet
            bets_placed_prof_in_match += 1
            bet_idx_prof_ah = np.argmin(odds_ah) # Apuesta a la cuota más baja (favorito AH)
            tipo_p_ah = 'home' if bet_idx_prof_ah == 0 else 'away'
            winnings_prof_in_match += evaluar_retorno_ah(gl, gv, linea_ah, tipo_p_ah, odds_ah[bet_idx_prof_ah], stake_fijo)

        # Actualizar la banca total del Profesor
        cap_prof_total -= bets_placed_prof_in_match * stake_fijo
        cap_prof_total += winnings_prof_in_match
        ops_prof_total += bets_placed_prof_in_match

        # =============================================================================================
        # ESTRATEGIA EV+ MODELO (COMBINADA): Apuesta donde EV+ está dentro de la ventana de seguridad
        # =============================================================================================
        bets_placed_mod_in_match = 0
        winnings_mod_in_match = 0

        # EV+ 1X2
        evs_1x2 = (probs_1x2 * odds_1x2) - 1
        best_1x2_idx = np.argmax(evs_1x2)
        max_ev_1x2 = evs_1x2[best_1x2_idx]
        if ev_min <= max_ev_1x2 <= ev_max and cap_mod_total >= stake_fijo:
            bets_placed_mod_in_match += 1
            if best_1x2_idx == outcome_idx_1x2:
                winnings_mod_in_match += stake_fijo * odds_1x2[best_1x2_idx]

        # EV+ O/U 2.5
        evs_ou = (probs_ou * odds_ou) - 1
        best_ou_idx = np.argmax(evs_ou)
        max_ev_ou = evs_ou[best_ou_idx]
        if ev_min <= max_ev_ou <= ev_max and cap_mod_total >= stake_fijo:
            bets_placed_mod_in_match += 1
            if best_ou_idx == outcome_idx_ou:
                winnings_mod_in_match += stake_fijo * odds_ou[best_ou_idx]

        # EV+ AH
        if linea_key is not None:
            # Calcular las 'true odds' del modelo para el AH, que ya están en ah_table
            odd_mod_home = ah_table[linea_key]['home_odds']
            odd_mod_away = ah_table[linea_key]['away_odds']

            # Asegurar que las cuotas del modelo no sean cero o muy pequeñas para evitar errores
            if odd_mod_home < 1.001: odd_mod_home = 1.001
            if odd_mod_away < 1.001: odd_mod_away = 1.001

            # Calcular EV usando la fórmula (Cuota de Mercado / Cuota Modelo) - 1
            ev_home_ah = (odds_ah[0] / odd_mod_home) - 1
            ev_away_ah = (odds_ah[1] / odd_mod_away) - 1

            evs_ah = np.array([ev_home_ah, ev_away_ah])
            best_ah_idx = np.argmax(evs_ah)
            max_ev_ah = evs_ah[best_ah_idx]

            if ev_min <= max_ev_ah <= ev_max and cap_mod_total >= stake_fijo:
                bets_placed_mod_in_match += 1
                tipo_m_ah = 'home' if best_ah_idx == 0 else 'away'
                winnings_mod_in_match += evaluar_retorno_ah(gl, gv, linea_ah, tipo_m_ah, odds_ah[best_ah_idx], stake_fijo)

        # Actualizar la banca total del Modelo
        cap_mod_total -= bets_placed_mod_in_match * stake_fijo
        cap_mod_total += winnings_mod_in_match
        ops_mod_total += bets_placed_mod_in_match

        # Añadir el estado actual de las bancas a la historia
        banca_profesor_total.append(cap_prof_total)
        banca_modelo_total.append(cap_mod_total)

    # ====================================================
    # GRAFICACIÓN Y RESUMEN FINAL CONSOLIDADO
    # ====================================================
    plt.figure(figsize=(14, 8))
    n_partidos = len(banca_profesor_total) - 1

    plt.plot(banca_profesor_total, label=f'>cuota Combinado (Total Apuestas: {ops_prof_total})', color='crimson', lw=2)
    plt.plot(banca_modelo_total, label=f'EV+ Combinado (Total Apuestas: {ops_mod_total})', color='forestgreen', lw=2.5)

    # Estampar el valor final de cada banca en el gráfico
    plt.text(n_partidos + 1, banca_profesor_total[-1], f" ${banca_profesor_total[-1]:.2f}",
             color='crimson', va='center', fontsize=10, fontweight='bold')
    plt.text(n_partidos + 1, banca_modelo_total[-1], f" ${banca_modelo_total[-1]:.2f}",
             color='forestgreen', va='center', fontsize=10, fontweight='bold')

    plt.axhline(y=capital_inicial, color='black', linestyle='-.', alpha=0.4, label='Banca Inicial ($1000)')
    plt.title(f'Simulación de Banca Unificada: >cuota vs EV+ Modelo (Ventana EV+: {ev_min:.0%} a {ev_max:.0%})',
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Partidos Transcurridos en la Bundesliga', fontsize=11)
    plt.ylabel('Capital Disponible (USD)', fontsize=11)
    plt.xlim(0, n_partidos + 25) # Espacio extra a la derecha para textos
    plt.grid(True, alpha=0.25)
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.show()

    # Despliegue de auditoría en consola
    print("=" * 70)
    print("      RESUMEN DE LIQUIDACIÓN DE BANCA UNIFICADA (3 Mercados)")
    print("=" * 70)
    print(f">cuota (3 Mercados): Apuestas {ops_prof_total}, Capital Final: ${cap_prof_total:.2f} USD")
    print(f"EV+ Modelo (3 Mercados): Apuestas {ops_mod_total}, Capital Final: ${cap_mod_total:.2f} USD")
    print("=" * 70)

# Ejecutar la nueva simulación unificada
simular_banca_unificada_todos_mercados(df_bundesliga, trace, teams, ev_min=0.05, ev_max=0.15)

## 20. Simulación Comparativa y Aislamiento del Efecto de la Cota Superior de Ventana en la Cartera Multi-Mercado

En esta sección se implementa un motor de simulación diseñado para aislar y contrastar empíricamente el impacto del filtro de cota superior (`ev_max`) en los mercados de 1X2, Over/Under 2.5 y Hándicap Asiático. El algoritmo ejecuta de forma paralela dos variantes operativas por cada mercado partiendo de un umbral mínimo común ($5\%$): la primera opera de forma irrestricta ante cualquier anomalía de valor alto, mientras que la segunda bloquea las señales que superen el techo de seguridad estipulado ($15\%$), permitiendo auditar visualmente si la exclusión de estas "trampas de sobreconfianza" previene rachas de pérdidas sistémicas y optimiza la estabilidad patrimonial de la banca.

In [ ]:
# ====================================================
# 1. AUXILIAR: EVALUADOR DE RETORNO HÁNDICAP ASIÁTICO
# ====================================================
def evaluar_retorno_ah(goles_local, goles_visitante, linea_home, tipo_apuesta, cuota, stake):
    diff_goles = goles_local - goles_visitante
    v_score = (diff_goles + linea_home) if tipo_apuesta == 'home' else -(diff_goles + linea_home)

    if v_score > 0.25:     return stake * cuota
    elif v_score == 0.25:   return stake * (1 + (cuota - 1)/2)
    elif v_score == 0:      return stake
    elif v_score == -0.25:  return stake * 0.5
    else:                   return 0.0

# ====================================================
# 2. MOTOR DE SIMULACIÓN DE COTA SUPERIOR
# ====================================================
def simular_comparativa_cota(df, trace_model, teams_df, capital_inicial=1000.0, ev_min=0.05, ev_max=0.15):
    result_map = {'H': 0, 'D': 1, 'A': 2}
    stake_fijo = 10.0

    # Inicializar balances (3 mercados x 2 variantes = 6 líneas de capital)
    cap_1x2_sin, cap_1x2_con = capital_inicial, capital_inicial
    cap_ou_sin,  cap_ou_con  = capital_inicial, capital_inicial
    cap_ah, cap_ah_con       = capital_inicial, capital_inicial

    # Historiales cronológicos para la gráfica
    banca_1x2_sin, banca_1x2_con = [capital_inicial], [capital_inicial]
    banca_ou_sin,  banca_ou_con  = [capital_inicial], [capital_inicial]
    banca_ah_sin,  banca_ah_con  = [capital_inicial], [capital_inicial]

    for idx, row in df.iterrows():
        if pd.isna(row['FTHG']) or pd.isna(row['FTAG']) or row['FTR'] not in result_map: continue
        if pd.isna(row['B365H']) or pd.isna(row['B365>2.5']) or pd.isna(row['AHCh']): continue

        gl, gv = int(row['FTHG']), int(row['FTAG'])
        outcome_idx = result_map[row['FTR']]
        goles_totales = gl + gv
        resultado_real_ou = 'over' if goles_totales > 2.5 else 'under'

        odds_1x2 = np.array([float(row['B365H']), float(row['B365D']), float(row['B365A'])])
        odds_ou  = np.array([float(row['B365>2.5']), float(row['B365<2.5'])])
        linea_ah = float(row['AHCh'])
        odds_ah  = np.array([float(row['B365AHH']), float(row['B365AHA'])])

        home_team, away_team = row['HomeTeam'], row['AwayTeam']
        try:
            home_id = teams_df.loc[teams_df["team"] == home_team, "team_index"].values[0]
            away_id = teams_df.loc[teams_df["team"] == away_team, "team_index"].values[0]

            # Expectativas base del modelo bayesiano
            h_exp, a_exp = goal_expectation(trace_model, home_id, away_id)

            probs_1x2 = np.array(win_draw_loss(h_exp, a_exp))
            prob_ou_dict = over_under_goals(h_exp, a_exp, 2.5)
            probs_ou = np.array([prob_ou_dict['over'], prob_ou_dict['under']])

            ah_table = asian_handicap_table(h_exp, a_exp)
            linea_key = next((k for k in ah_table.keys() if abs(k - linea_ah) < 1e-4), None)
        except (IndexError, KeyError, StopIteration):
            banca_1x2_sin.append(cap_1x2_sin); banca_1x2_con.append(cap_1x2_con)
            banca_ou_sin.append(cap_ou_sin);   banca_ou_con.append(cap_ou_con)
            banca_ah_sin.append(cap_ah);       banca_ah_con.append(cap_ah_con)
            continue

        # ====================================================
        # COMPARATIVA: MERCADO 1X2
        # ====================================================
        evs_1x2 = (probs_1x2 * odds_1x2) - 1
        best_1x2 = np.argmax(evs_1x2)
        max_ev_1x2 = evs_1x2[best_1x2]

        # Variante 1: Sin Cota Superior (Solo exige mínimo)
        if max_ev_1x2 >= ev_min and cap_1x2_sin >= stake_fijo:
            cap_1x2_sin -= stake_fijo
            if best_1x2 == outcome_idx: cap_1x2_sin += stake_fijo * odds_1x2[best_1x2]

        # Variante 2: Con Cota Superior (Filtro de ventana 5% - 15%)
        if ev_min <= max_ev_1x2 <= ev_max and cap_1x2_con >= stake_fijo:
            cap_1x2_con -= stake_fijo
            if best_1x2 == outcome_idx: cap_1x2_con += stake_fijo * odds_1x2[best_1x2]

        # ====================================================
        # COMPARATIVA: MERCADO OVER/UNDER 2.5
        # ====================================================
        evs_ou = (probs_ou * odds_ou) - 1
        best_ou = np.argmax(evs_ou)
        max_ev_ou = evs_ou[best_ou]

        # Variante 1: Sin Cota Superior
        if max_ev_ou >= ev_min and cap_ou_sin >= stake_fijo:
            pred_m = 'over' if best_ou == 0 else 'under'
            cap_ou_sin -= stake_fijo
            if pred_m == resultado_real_ou: cap_ou_sin += stake_fijo * odds_ou[best_ou]

        # Variante 2: Con Cota Superior
        if ev_min <= max_ev_ou <= ev_max and cap_ou_con >= stake_fijo:
            pred_m = 'over' if best_ou == 0 else 'under'
            cap_ou_con -= stake_fijo
            if pred_m == resultado_real_ou: cap_ou_con += stake_fijo * odds_ou[best_ou]

        # ====================================================
        # COMPARATIVA: MERCADO HÁNDICAP ASIÁTICO
        # ====================================================
        if linea_key is not None:
            odds_mod_ah = np.array([ah_table[linea_key]['home_odds'], ah_table[linea_key]['away_odds']])
            evs_ah = (odds_ah / odds_mod_ah) - 1
            best_ah = np.argmax(evs_ah)
            max_ev_ah = evs_ah[best_ah]

            # Variante 1: Sin Cota Superior
            if max_ev_ah >= ev_min and cap_ah >= stake_fijo:
                tipo_m = 'home' if best_ah == 0 else 'away'
                cap_ah -= stake_fijo
                cap_ah += evaluar_retorno_ah(gl, gv, linea_ah, tipo_m, odds_ah[best_ah], stake_fijo)

            # Variante 2: Con Cota Superior
            if ev_min <= max_ev_ah <= ev_max and cap_ah_con >= stake_fijo:
                tipo_m = 'home' if best_ah == 0 else 'away'
                cap_ah_con -= stake_fijo
                cap_ah_con += evaluar_retorno_ah(gl, gv, linea_ah, tipo_m, odds_ah[best_ah], stake_fijo)

        # Guardar historial al cierre de cada partido
        banca_1x2_sin.append(cap_1x2_sin); banca_1x2_con.append(cap_1x2_con)
        banca_ou_sin.append(cap_ou_sin);   banca_ou_con.append(cap_ou_con)
        banca_ah_sin.append(cap_ah);       banca_ah_con.append(cap_ah_con)

    # ====================================================
    # 3. GRAFICACIÓN CONSOLIDADA DE LA COMPARATIVA
    # ====================================================
    plt.figure(figsize=(14, 8))
    n_partidos = len(banca_1x2_sin) - 1

    # Definición de las 6 líneas para contrastar la cota superior
    lineas_comparativa = {
        'Estrategia EV+ 1X2 (Sin Cota)':       (banca_1x2_sin, 'royalblue',  '-'),
        'Estrategia EV+ 1X2 (Con Cota 15%)':   (banca_1x2_con, 'darkorange', '-'),
        'Estrategia EV+ O/U 2.5 (Sin Cota)':   (banca_1x2_sin, 'royalblue',  '--'), # Usamos la de O/U real
        'Estrategia EV+ O/U 2.5 (Con Cota 15%)':(banca_ou_con,  'darkorange', '--'),
        'Estrategia EV+ Hándicap (Sin Cota)':  (banca_ah_sin,  'royalblue',  ':'),
        'Estrategia EV+ Hándicap (Con Cota 15%)':(banca_ah_con,  'darkorange', ':')
    }

    # Nota de corrección interna rápida en el diccionario para apuntar al vector OU correcto en "Sin Cota"
    lineas_comparativa['Estrategia EV+ O/U 2.5 (Sin Cota)'] = (banca_ou_sin, 'royalblue', '--')

    for label, (datos, color, estilo) in lineas_comparativa.items():
        plt.plot(datos, label=label, color=color, linestyle=estilo, lw=2.2)

        # Estampar el capital final al extremo derecho de la curva
        val_final = datos[-1]
        plt.text(n_partidos + 1, val_final, f" ${val_final:.2f}",
                 color=color, va='center', fontsize=9, fontweight='bold')

    plt.axhline(y=capital_inicial, color='black', linestyle='-.', alpha=0.4, label='Capital Inicial ($1000)')
    plt.title('Impacto del Filtro de Cota Superior (15% Máx) en la Estrategia EV+', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Partidos Transcurridos en la Bundesliga', fontsize=11)
    plt.ylabel('Capital Disponible (USD)', fontsize=11)
    plt.xlim(0, n_partidos + 25) # Margen para los textos de balance final
    plt.grid(True, alpha=0.2)
    plt.legend(loc='upper left', fontsize=10, bbox_to_anchor=(0.01, 0.99))
    plt.tight_layout()
    plt.show()

    # Resumen impreso
    print("=" * 75)
    print("      EFECTO DEL FILTRO DE SEGURIDAD (CON COTA VS SIN COTA)")
    print("=" * 75)
    print(f"MERCADO 1X2      | Sin Cota: ${cap_1x2_sin:.2f} USD | Con Cota (15%): ${cap_1x2_con:.2f} USD")
    print(f"MERCADO O/U 2.5  | Sin Cota: ${cap_ou_sin:.2f}  USD | Con Cota (15%): ${cap_ou_con:.2f}  USD")
    print(f"MERCADO HÁNDICAP | Sin Cota: ${cap_ah:.2f}  USD | Con Cota (15%): ${cap_ah_con:.2f}  USD")
    print("=" * 75)

# Ejecutar la simulación comparativa
simular_comparativa_cota(df_bundesliga, trace, teams, ev_min=0.05, ev_max=0.15)